In [0]:
import mlflow
import mlflow.deployments
from databricks.sdk import WorkspaceClient

context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
host = context.apiUrl().get()
token = context.apiToken().get()

w = WorkspaceClient(host=host, token=token)
deploy_client = mlflow.deployments.get_deploy_client("databricks")

index_name = "dbr_dev_trial.artemzharkov10_gold.knowledge_base_index"
endpoint_model = "databricks-meta-llama-3-3-70b-instruct"
question = "What is the braking distance on wet asphalt and what is hydroplaning?"

In [0]:
response_no_rag = deploy_client.predict(
    endpoint=endpoint_model,
    inputs={
        "messages": [{"role": "user", "content": question}],
        "max_tokens": 512,
        "temperature": 0.1
    }
)
answer_no_rag = response_no_rag['choices'][0]['message']['content']


In [0]:

search_results = w.vector_search_indexes.query_index(
    index_name=index_name,
    query_text=question,
    columns=["text_content"],
    num_results=3
)

context_chunks = [row[0] for row in search_results.result.data_array] if search_results.result else []
context_text = "\n\n".join(context_chunks)

system_prompt = f"Answer the user's question based ONLY on the following context:\n{context_text}"

response_rag = deploy_client.predict(
    endpoint=endpoint_model,
    inputs={
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ],
        "max_tokens": 512,
        "temperature": 0.1
    }
)
answer_rag = response_rag['choices'][0]['message']['content']

eval_prompt = f"""
Evaluate the following RAG answer for hallucinations and missing context. 
Context provided to the model: {context_text}
Model's answer: {answer_rag}

Provide a score from 1 to 5 for:
1. Relevance (Does it answer the prompt?)
2. Factuality (Are all facts strictly from the context without external hallucinations?)
"""
response_eval = deploy_client.predict(
    endpoint=endpoint_model,
    inputs={"messages": [{"role": "user", "content": eval_prompt}], "max_tokens": 256}
)
evaluation_result = response_eval['choices'][0]['message']['content']

mlflow.set_experiment("/Shared/RAG_Safety_QA_Monitoring")

with mlflow.start_run(run_name="RAG_Comparison_and_Eval"):

    mlflow.log_param("model_used", endpoint_model)
    mlflow.log_param("question", question)

    mlflow.log_text(context_text, "retrieved_context.txt")
    mlflow.log_text(answer_no_rag, "answer_baseline_no_rag.txt")
    mlflow.log_text(answer_rag, "answer_rag.txt")
    mlflow.log_text(evaluation_result, "evaluation_report.txt")

print(f"\n[LLM]:\n{evaluation_result}")